In [2]:
# Run this if facing issue with transformers
# pip install torch torchvision torchaudio transformers

import torch
from datasets import Dataset
from scipy.stats import loguniform
import optuna
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification, pipeline, BertForSequenceClassification, BertTokenizer
import torch

from Utils.import_packages import *

In [3]:
train_df = pd.read_csv('Data/train_labelled.csv')
val_df = pd.read_csv('Data/val_labelled.csv')
test_df = pd.read_csv('Data/test_labelled.csv')

In [12]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

ind = []
for i, s in enumerate(train_df["title"].tolist()):
    scores = analyzer.polarity_scores(s)
    if scores["neu"] == max(scores["neg"], scores["neu"], scores["pos"]):
        ind += [i]
train_df = train_df.drop(index=ind)
print(train_df.shape)

ind = []
for i, s in enumerate(val_df["title"].tolist()):
    scores = analyzer.polarity_scores(s)
    if scores["neu"] == max(scores["neg"], scores["neu"], scores["pos"]):
        ind += [i]
val_df = val_df.drop(index=ind)
print(val_df.shape)

ind = []
for i, s in enumerate(test_df["title"].tolist()):
    scores = analyzer.polarity_scores(s)
    if scores["neu"] == max(scores["neg"], scores["neu"], scores["pos"]):
        ind += [i]

test_df = test_df.drop(index=ind)
print(test_df.shape)

(825, 6)
(46, 6)
(390, 6)


In [13]:
# Apply preprocessing to datasets
train_df["title"] = train_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)
val_df["title"] = val_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)
test_df["title"] = test_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)

train_df["title"] = train_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())
val_df["title"] = val_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())
test_df["title"] = test_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())

train_df = train_df.sort_values(by="date")
val_df = val_df.sort_values(by="date")
test_df = test_df.sort_values(by="date")

In [14]:
# Check distribution of labels
print(train_df['sentiment_label'].value_counts())
print(val_df['sentiment_label'].value_counts())
print(test_df['sentiment_label'].value_counts())

sentiment_label
1    414
0    411
Name: count, dtype: int64
sentiment_label
1    23
0    23
Name: count, dtype: int64
sentiment_label
0    255
1    135
Name: count, dtype: int64


# BERT and FinBERT
BERT and FinBERT are a pre-trained text analysis models, with Finbert particularly trained on financial data. It labels financial data as either "positive", "neutral" or "negative" by assigning a probability to each class, and return the class with the highest probability.
<br> For our use case, we labelled sentiments using only '1' or '0' for the excess 3 day aggregated returns.
<br> For BERT, we will fit the model directly on our test set, since it is pre-trained, just to see how it performs as a baseline compared to FinBERT. This is done by predicting the 3 sentiment probabilities, then using them to map to 1 or 0 based on whether P(positive) >  P(negative)
<BR> For FinBERT, we will first fine tune finBERT to label using 2 classifications on our training set, then using Optuna to search for the best parameters that will return the model with the highest F1 by evaluating its performance on the valuation set. This fine tuned model is then fitted to test set and sentiment labels are predicted again based on the similar mapping logic as before.

## BERT
### Fitting on Test Set

In [16]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Load the saved model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3).to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model.eval()

# Preprocessing function for BERT
def preprocess_text(text_list):
    return [text.strip().lower() for text in text_list]

# Tokenize data
def preprocess_data(example):
    processed_texts = preprocess_text(example['title'])
    return tokenizer(processed_texts, padding='max_length', truncation=True, max_length=128)

# Tokenize the test set using the saved tokenizer
test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df = test_tokenized_df.rename_column("sentiment_label", "labels")
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Run predictions on the test set
all_logits = []
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tokenized_df, batch_size=8):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Save predictions to test DataFrame
test_df_results_bert = test_df.copy()
test_df_results_bert['prob_negative'] = prob_negative
test_df_results_bert['prob_neutral'] = prob_neutral
test_df_results_bert['prob_positive'] = prob_positive
test_df_results_bert['finbert_sentiment_labels'] = predicted_labels

# Evaluate metrics if ground-truth labels exist
if 'sentiment_label' in test_df_results_bert.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df_results_bert['sentiment_label'], predicted_labels, average='weighted'
    )
    accuracy = accuracy_score(test_df_results_bert['sentiment_label'], predicted_labels)

    print(f"Test Accuracy: {accuracy}")
    print(f"Test Precision: {precision}")
    print(f"Test Recall: {recall}")
    print(f"Test F1-Score: {f1}")

test_df_results_bert

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/390 [00:00<?, ? examples/s]

Test Accuracy: 0.6538461538461539
Test Precision: 0.42751479289940825
Test Recall: 0.6538461538461539
Test F1-Score: 0.516994633273703


C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv2\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,title,source,topic,company name(s) - cleaned,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
58421,Visa Reimagines Customer Loyalty with New Web3...,Business Wire,Transaction and Payment Processing Services,Visa Inc.,2024-01-04,1,0.219881,0.481068,0.299052,0
43862,True Potential Appoints Northern Trust for Ass...,Business Wire,Asset Management and Custody Banks,Northern Trust Corporation,2024-01-19,0,0.225078,0.475394,0.299528,0
60612,Wells Fargo & Company Approves and Adopts the ...,SEC Form 8k,Diversified Banks,Wells Fargo & Company,2024-01-24,1,0.212528,0.512595,0.274877,0
642,Abbott Laboratories Enters into Five Year Cred...,SEC Form 8k,Health Care Equipment,Abbott Laboratories,2024-01-29,1,0.211281,0.505997,0.282722,0
35225,Leidos Wins Contract to Support Advanced Resea...,PR Newswire,Research and Consulting Services,"Leidos Holdings, Inc.",2024-01-29,0,0.227047,0.477124,0.295828,0
...,...,...,...,...,...,...,...,...,...,...
31428,What s More Wicked than the Crime of,https://mises.org/podcasts/human-action-podcas...,economics,NaN,2024-12-13,0,0.246392,0.432421,0.321187,0
3589,Nice for them nice for your wallet,https://www.komando.com/tips/nice-for-them-nic...,tech,NaN,2024-12-13,0,0.258311,0.421892,0.319798,0
35218,The Missing Secret,https://mises.org/friday-philosophy/missing-se...,economics,NaN,2024-12-13,0,0.240583,0.427770,0.331647,0
21538,What s More Wicked than the Crime of,https://mises.org/podcasts/human-action-podcas...,economics,NaN,2024-12-13,0,0.246392,0.432421,0.321187,0


In [18]:
test_df_results_bert.to_csv('results/test_df_results_bert_filtered.csv', index=False)

## Prosus AI/FinBERT
### Optuna Hyperparameter Optimisation

In [19]:
train_subset_size = int(len(train_df) * 0.05) # take 5% of train set

# Sample proportionally across all dates
train_subset_df = train_df.iloc[::len(train_df) // train_subset_size]
train_subset_df = train_subset_df.sort_values(by=['date']).reset_index(drop=True)

In [20]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# Tokenize data
def preprocess_data(example):
    return tokenizer(example['title'], padding='max_length', max_length=128, truncation=True)

train_subset_tokenized_df = Dataset.from_pandas(train_subset_df).map(preprocess_data, batched=True)
train_subset_tokenized_df = train_subset_tokenized_df.rename_column("sentiment_label", "labels")
train_subset_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df = val_tokenized_df.rename_column("sentiment_label", "labels")
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Define metrics for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    prob_negative = probabilities[:, 1]
    prob_neutral = probabilities[:, 2]
    prob_positive = probabilities[:, 0]

    predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average='weighted')
    accuracy = accuracy_score(labels, predicted_labels)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Define the Optuna objective function
def objective(trial):
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    num_train_epochs = trial.suggest_int('num_train_epochs', 2, 5)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=3).to(device)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir="./train_subset_results",
        eval_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_dir="./train_subset_logs",
        logging_steps=10,
        disable_tqdm=False,
        load_best_model_at_end=True
    )

    # Define the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset_tokenized_df,
        eval_dataset=val_tokenized_df,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Train model
    trainer.train()

    # Evaluate model
    eval_results = trainer.evaluate()
    return eval_results['eval_f1']

# Initialize an Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Output the best hyperparameters
print("Best hyperparameters:")
print(study.best_params)

Using device: cuda


Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

[I 2024-12-17 23:43:15,012] A new study created in memory with name: no-name-c508629f-19de-4799-98c9-4faaf3745192
C:\Users\Tylus\AppData\Local\Temp\ipykernel_5748\1610795840.py:46: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
C:\Users\Tylus\AppData\Local\Temp\ipykernel_5748\1610795840.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,2.694781,0.500000,0.500000,0.500000,0.499764
2,No log,2.334510,0.478261,0.478261,0.478261,0.478261
3,No log,2.137358,0.586957,0.587121,0.586957,0.586761
4,No log,2.073454,0.586957,0.587121,0.586957,0.586761


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,2.109188,0.565217,0.567251,0.565217,0.561905
2,No log,1.774052,0.565217,0.574194,0.565217,0.551657
3,No log,1.594790,0.608696,0.612086,0.608696,0.605714


[I 2024-12-17 23:48:49,934] Trial 48 finished with value: 0.6057142857142858 and parameters: {'learning_rate': 2.2655006167161002e-05, 'batch_size': 16, 'num_train_epochs': 3}. Best is trial 45 with value: 0.7106918238993711.
C:\Users\Tylus\AppData\Local\Temp\ipykernel_5748\1610795840.py:46: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
C:\Users\Tylus\AppData\Local\Temp\ipykernel_5748\1610795840.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,2.494835,0.478261,0.478261,0.478261,0.478261
2,No log,2.085882,0.586957,0.587121,0.586957,0.586761
3,No log,2.020228,0.630435,0.630682,0.630435,0.630260


[I 2024-12-17 23:48:56,021] Trial 49 finished with value: 0.6302600472813239 and parameters: {'learning_rate': 1.6018091621125056e-05, 'batch_size': 32, 'num_train_epochs': 3}. Best is trial 45 with value: 0.7106918238993711.


Best hyperparameters:
{'learning_rate': 2.1730982555731134e-05, 'batch_size': 32, 'num_train_epochs': 3}


### Fine Tuning on Full Train Set

In [21]:
# Tokenize data
train_tokenized_df = Dataset.from_pandas(train_df).map(preprocess_data, batched=True)
train_tokenized_df = train_tokenized_df.rename_column("sentiment_label", "labels")
train_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df = val_tokenized_df.rename_column("sentiment_label", "labels")
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Train and evaluate the model using the best hyperparameters from Optuna
best_params = study.best_params

model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=3).to(device)

training_args = TrainingArguments(
    output_dir="./train_results",
    evaluation_strategy="epoch",
    learning_rate=best_params['learning_rate'],
    per_device_train_batch_size=best_params['batch_size'],
    num_train_epochs=best_params['num_train_epochs'],
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./train_logs",
    logging_steps=10,
    disable_tqdm=False,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_df,
    eval_dataset=val_tokenized_df,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Save trained model
output_dir = "./finbert_finetuned_filtered"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Final evaluation on the validation set
model.eval()
all_logits = []

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(val_tokenized_df, batch_size=best_params['batch_size']):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

# Extract probabilities for each class
prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

# Transform probabilities into labels
predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Add probabilities and predictions to the validation DataFrame
val_df['prob_negative'] = prob_negative
val_df['prob_neutral'] = prob_neutral
val_df['prob_positive'] = prob_positive
val_df['finbert_sentiment_labels'] = predicted_labels

# Final evaluation metrics
if 'sentiment_label' in val_df.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(val_df['sentiment_label'], predicted_labels, average='weighted')
    accuracy = accuracy_score(val_df['sentiment_label'], predicted_labels)

    print(f"Validation Accuracy: {accuracy}")
    print(f"Validation Precision: {precision}")
    print(f"Validation Recall: {recall}")
    print(f"Validation F1-Score: {f1}")

val_df

Map:   0%|          | 0/825 [00:00<?, ? examples/s]

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv2\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Tylus\AppData\Local\Temp\ipykernel_5748\2610867238.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.845900,0.768583,0.456522,0.456439,0.456522,0.456265
2,0.700200,0.783781,0.608696,0.623656,0.608696,0.596491
3,0.675600,0.814824,0.652174,0.656920,0.652174,0.649524


Validation Accuracy: 0.45652173913043476
Validation Precision: 0.4564393939393939
Validation Recall: 0.45652173913043476
Validation F1-Score: 0.4562647754137116


,title,source,topic,company name(s) - cleaned,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
31264,Provence Beauty Debuts Exclusively At Ulta Beauty,PR Newswire,Other Specialty Retail,"Ulta Beauty, Inc.",2023-01-04,1,0.560422,0.026716,0.412861,0
1203,American International Group Seeks Inorganic G...,Transcript Collection,Multi-line Insurance,"American International Group, Inc.",2023-02-16,1,0.492326,0.024783,0.482891,0
27522,"Rollins, Inc. Announces New Credit Agreement",PR Newswire,Environmental and Facilities Services,"Rollins, Inc.",2023-02-27,0,0.594066,0.025744,0.380190,0
28241,"Super Micro Computer, Inc. - Special Call",Company Website,"Technology Hardware, Storage and Peripherals","Super Micro Computer, Inc.",2023-03-09,0,0.460125,0.041321,0.498554,1
15390,Hershey's KISSES Announces to Introduce New He...,PR Newswire,Packaged Foods and Meats,The Hershey Company,2023-03-15,1,0.503770,0.024260,0.471970,0
12413,Fair Isaac Corporation - Special Call,Business Wire,Application Software,Fair Isaac Corporation,2023-03-30,0,0.415581,0.037775,0.546644,1
2463,Aon More Actively Considering M&A Opportunities,Other,Insurance Brokers,Aon plc,2023-03-31,1,0.501279,0.029584,0.469137,0
10030,DTE Energy Company - Special Call,Company Website,Multi-Utilities,DTE Energy Company,2023-04-03,1,0.433389,0.036526,0.530085,1
23878,Northern Trust to Provide Asset Servicing Solu...,Business Wire,Asset Management and Custody Banks,Northern Trust Corporation,2023-04-11,1,0.315626,0.017073,0.667300,1
17262,Georgia United Credit Union Strengthens Digita...,PR Newswire,Transaction and Payment Processing Services,"Jack Henry & Associates, Inc.",2023-04-12,0,0.502797,0.026640,0.470563,0


### Fitting on Test Set

In [23]:
# Load the saved model and tokenizer
output_dir = "./finbert_finetuned_filtered"
model = AutoModelForSequenceClassification.from_pretrained(output_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model.eval()

# Tokenize the test set using the saved tokenizer
test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df = test_tokenized_df.rename_column("sentiment_label", "labels")
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Run predictions on the test set
all_logits = []
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tokenized_df, batch_size=best_params['batch_size']):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Save predictions to test DataFrame
test_df_results = test_df.copy()
test_df_results['prob_negative'] = prob_negative
test_df_results['prob_neutral'] = prob_neutral
test_df_results['prob_positive'] = prob_positive
test_df_results['finbert_sentiment_labels'] = predicted_labels

# Evaluate metrics if ground-truth labels exist
if 'sentiment_label' in test_df_results.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df_results['sentiment_label'], predicted_labels, average='weighted'
    )
    accuracy = accuracy_score(test_df_results['sentiment_label'], predicted_labels)

    print(f"Test Accuracy: {accuracy}")
    print(f"Test Precision: {precision}")
    print(f"Test Recall: {recall}")
    print(f"Test F1-Score: {f1}")

test_df_results

Map:   0%|          | 0/390 [00:00<?, ? examples/s]

Test Accuracy: 0.5974358974358974
Test Precision: 0.6081089743589744
Test Recall: 0.5974358974358974
Test F1-Score: 0.6019302335091807


,title,source,topic,company name(s) - cleaned,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
58421,Visa Reimagines Customer Loyalty with New Web3...,Business Wire,Transaction and Payment Processing Services,Visa Inc.,2024-01-04,1,0.423700,0.021572,0.554728,1
43862,True Potential Appoints Northern Trust for Ass...,Business Wire,Asset Management and Custody Banks,Northern Trust Corporation,2024-01-19,0,0.455273,0.023059,0.521668,1
60612,Wells Fargo & Company Approves and Adopts the ...,SEC Form 8k,Diversified Banks,Wells Fargo & Company,2024-01-24,1,0.507237,0.024754,0.468009,0
642,Abbott Laboratories Enters into Five Year Cred...,SEC Form 8k,Health Care Equipment,Abbott Laboratories,2024-01-29,1,0.788146,0.014926,0.196928,0
35225,Leidos Wins Contract to Support Advanced Resea...,PR Newswire,Research and Consulting Services,"Leidos Holdings, Inc.",2024-01-29,0,0.437296,0.020836,0.541868,1
...,...,...,...,...,...,...,...,...,...,...
31428,What s More Wicked than the Crime of,https://mises.org/podcasts/human-action-podcas...,economics,NaN,2024-12-13,0,0.648875,0.140803,0.210322,0
3589,Nice for them nice for your wallet,https://www.komando.com/tips/nice-for-them-nic...,tech,NaN,2024-12-13,0,0.533973,0.044016,0.422011,0
35218,The Missing Secret,https://mises.org/friday-philosophy/missing-se...,economics,NaN,2024-12-13,0,0.664648,0.110303,0.225049,0
21538,What s More Wicked than the Crime of,https://mises.org/podcasts/human-action-podcas...,economics,NaN,2024-12-13,0,0.648875,0.140803,0.210322,0


In [24]:
# Save predictions to the test dataframe
test_df_results.to_csv("results/test_df_results_filtered.csv", index=False)